# Buy the Fear, Sell the Greed
## Alpaca data → strategy signals → vectorbt trades → performance matrix

**How to use:** run Setup once, enter your credentials when prompted, set ticker(s) and dates, then run any strategy section. Each section has its own input cell. Use **Run All** to compare all documented strategies in this book. You do not need to edit YAML or package code.

**Local:** extract the project ZIP, open this notebook inside `notebooks/`, and use Python 3.12. **Colab:** upload this notebook, then upload the project ZIP when Setup asks. The same notebook works in both environments.

[Strategy reference](https://docs.google.com/document/d/1MzbEHZr38_fwsD82O456i8YsoCz5A87QVm7S5Gl660Y/edit). Source-close results are idealized; next-open is an explicitly different timing assumption. This notebook defaults to real Alpaca data, never synthetic fallback.

### Contents
1. RSI PowerZones
2. CRASH
3. Vol Panics
4. VXX Trend
5. Trading New Highs
6. TPS — Time, Price, Scale-in
7. Terror Gaps — specification blocked

## 0. Setup — install the shared project

Run this cell once per fresh runtime. If NumPy/pandas were imported before installation and changed versions, restart the runtime and rerun Setup. Only Python 3.12 has been runtime-tested. The code lives in the shared package; these notebooks contain usage examples, not duplicated trading logic.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, zipfile, io

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path("/content/larryconnors-trading")
    if not (PROJECT / "pyproject.toml").exists():
        from google.colab import files
        print("Upload the provided larryconnors-trading.zip project archive.")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP")
        with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
            for name in archive.namelist():
                path = Path(name)
                if path.is_absolute() or ".." in path.parts or path.parts[0] != "larryconnors-trading":
                    raise ValueError("Unexpected archive layout")
            archive.extractall("/content")
        del uploaded
else:
    PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists() and (p / "src/connors").exists()), None)
    if PROJECT is None:
        raise RuntimeError("Open this notebook from inside the extracted project folder")

assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.12 (tested)"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-c", str(PROJECT / "constraints.txt"), "-e", str(PROJECT) + "[dev]"])
sys.path.insert(0, str(PROJECT / "src"))
print("Project ready:", PROJECT)


### Alpaca credentials and imports

Set **API_KEY** and **API_SECRET** in your local environment or in Colab **Secrets** (enable notebook access). Otherwise, hidden input prompts appear. Keys stay in memory and are never printed or written into reports. No trading orders are submitted.

The package uses `StockHistoricalDataClient.get_stock_bars(StockBarsRequest(..., timeframe=TimeFrame.Day))`. SDK pagination retrieves all pages; the adapter caches bars across strategies and downloads warm-up history before your start date.

In [ ]:
# The official Alpaca SDK supplies historical data; vectorbt runs the portfolios.
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from connors.alpaca_sdk import read_notebook_secret
from connors.notebook import NotebookResearch, performance_matrix, display_performance_matrix
from connors.notebook_strategies import (
    PositionType, high_low_strategy, rsi_25_75_strategy, r3_strategy,
    percent_b_strategy, multiple_days_strategy, rsi_sma_strategy, tps_strategy,
    rsi_under_5_strategy, cumulative_rsi_strategy, double_seven_strategy,
    vix_rsi_strategy, trin_strategy, sp_short_strategy, ten_day_low_strategy,
    overnight_strategy, powerzones_strategy, crash_strategy, vol_panics_strategy,
    vxx_trend_strategy, new_highs_strategy, terror_gaps_strategy,
)

# Environment variables, Colab Secrets, or hidden interactive prompts.
# Do not replace these calls with literal secrets in saved cells.
API_KEY = read_notebook_secret("API_KEY")
API_SECRET = read_notebook_secret("API_SECRET")
client = StockHistoricalDataClient(API_KEY, API_SECRET)
del API_KEY, API_SECRET  # the SDK client retains its runtime credentials


### Symbols, dates, costs and execution defaults

Your backtest starts on `start_date`; earlier bars only initialize indicators. Supply a string such as `"SPY"` or a list such as `["SPY", "QQQ"]`. Each ticker gets its own initial capital. The matrix compares independently funded runs, not a shared-cash portfolio.

`position_steps` expresses fractions of a planned full position, e.g. `[0.5, 0.5]` for two equal units or `[0.1, 0.2, 0.3, 0.4]` for TPS. Source equal-unit systems buy equal **shares**, so later dollar notionals can differ. `position_fraction=0.95` reserves room for costs; it can be changed per call. Unsupported extra stages are rejected, not invented.

Short sections below explicitly assume borrow availability; set the annual `BORROW_RATE` to your estimate. IEX is not consolidated SIP and volume-based screens can differ. Split-adjusted research prices exclude dividends; dollar-price screens on adjusted data are an approximation. A custom ticker is a user-selected universe experiment, not an automatic SPX/VIX proxy.

In [ ]:
from datetime import date
SYMBOLS = ["SPY"]             # "SPY" or ["SPY", "QQQ", "IWM"]
start_date = "2022-01-03"    # first date of the backtest (warm-up is fetched automatically)
end_date = str(date.today())      # inclusive last date; choose completed historical sessions
FEED = "iex"                # "sip" if your Alpaca entitlement permits it
INITIAL_CAPITAL = 100_000    # independent capital PER ticker
EXECUTION_MODE = "source_close"  # "next_open" is a timing variant
ASSUME_BORROW_AVAILABLE = True   # explicit research assumption for short sections
BORROW_RATE = 0.0                # annual decimal; choose a nonzero estimate if appropriate

research = NotebookResearch(
    client, cache=PROJECT / ".cache/alpaca_sdk", output=PROJECT / "reports/notebooks",
    feed=FEED, adjustment="split", initial_capital=INITIAL_CAPITAL,
    execution_mode=EXECUTION_MODE, fees=0.0005, slippage=0.0005,
    assume_borrow_available=ASSUME_BORROW_AVAILABLE, borrow_rate=BORROW_RATE,
)
plot_all_stocks_pnl_charts = research.plot_all_stocks_pnl_charts
results = {}  # collected runs for the performance matrix at the end


### Reading the outputs

Each run returns a `NotebookBatch` with `.results[symbol].portfolio` (vectorbt), `.figures[symbol]`, and `.performance` (DataFrame). Charts show completed winning/losing positions, P&L, daily equity and drawdown. Scale-ins count as one position. Open final positions affect equity but not closed-trade win rate. Saved HTML charts and CSV ledgers are under `reports/notebooks/`.

The performance matrix uses daily-return Sharpe (252 sessions/year, zero risk-free), calendar-time CAGR for runs at least one year, net episode return divided by entered notional, and two-way turnover. Avoid treating vectorbt's internal three-phase event frequency as daily returns; the matrix uses actual daily closes.

## 1. RSI PowerZones

**Source readiness:** ready. 

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** close > SMA200 AND RSI4 < entry
- **Additions:** one equal unit on later RSI4 < add
- **Exit:** RSI4 > 55
- **Default parameters:** `{"entry": 30, "add": 25, "exit": 55}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'RSI PowerZones'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.powerzones'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[0.5, 0.5],
    strategy_func=powerzones_strategy,
)


### PowerZones 25/20 alternative

This is the same RSI(4) rule as aggressive HPE RSI25 long, with a second unit below20 and an exit above55. It remains distinct from RSI(2) systems.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'HPE RSI 25 & RSI 75'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['powerzones25'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[0.5, 0.5],
    strategy_func=rsi_25_75_strategy,
    variant='powerzones25',
)


## 2. CRASH

**Source readiness:** experimental. ConnorsRSI(3,2,100), strict less-than percent rank and HV conventions are explicit assumptions absent in document

**Universe:** Stocks; historical point-in-time universe and delisted membership must be supplied

### Short rules

- **Entry:** close > 5 AND mean(volume,21) >= 1000000 AND HV100 >= hv_min AND ConnorsRSI >= 90; next-day sell limit close*(1+limit_pct)
- **Additions:** none
- **Exit:** ConnorsRSI < exit
- **Default parameters:** `{"limit_pct": 0.03, "exit": 30, "hv_min": 1.0}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'CRASH'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.crash'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.SHORT,
    position_steps=[1.0],
    strategy_func=crash_strategy,
    allow_experimental=True,  # explicit assumptions explained above
)


### Named variants — optional comparison

Each item below preserves its documented thresholds and scale-in rules. Set `RUN_VARIANTS = True` to run them on the same dates. Keep parameter exploration in training/validation data; do not choose the best variant on a final test period.

- `crash5`: `{"limit_pct": 0.05, "exit": 30, "hv_min": 1.0}`
- `crash_exit20`: `{"limit_pct": 0.03, "exit": 20, "hv_min": 1.0}`
- `crash_hv60`: `{"limit_pct": 0.03, "exit": 30, "hv_min": 0.6}`
- `crash_hv80`: `{"limit_pct": 0.03, "exit": 30, "hv_min": 0.8}`

In [ ]:
RUN_VARIANTS = False
if RUN_VARIANTS:
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'CRASH'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['crash5'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=crash_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='crash5',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'CRASH'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['crash_exit20'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=crash_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='crash_exit20',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'CRASH'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['crash_hv60'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=crash_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='crash_hv60',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'CRASH'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['crash_hv80'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=crash_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='crash_hv80',
    )


## 3. Vol Panics

**Source readiness:** ready. 

**Universe:** VXX; historical note/product changes disclosed

### Short rules

- **Entry:** close > SMA5 AND RSI4 > 70
- **Additions:** basic 100%; double 50/50 equal shares above first actual entry; staged 20/30/50 or 10/20/30/40 notional above latest actual entry
- **Exit:** close < SMA5, or named RSI4 < 20 variant
- **Default parameters:** `{"stages": [1.0], "exit_kind": "sma"}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Vol Panics'
symbols = ["VXX"]  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.vol_panics'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.SHORT,
    position_steps=[1.0],
    strategy_func=vol_panics_strategy,
)


### Named variants — optional comparison

Each item below preserves its documented thresholds and scale-in rules. Set `RUN_VARIANTS = True` to run them on the same dates. Keep parameter exploration in training/validation data; do not choose the best variant on a final test period.

- `vol_double`: `{"stages": [0.5, 0.5], "exit_kind": "sma", "equal_units": true}`
- `vol235`: `{"stages": [0.2, 0.3, 0.5], "exit_kind": "sma"}`
- `vol1234`: `{"stages": [0.1, 0.2, 0.3, 0.4], "exit_kind": "sma"}`
- `vol_rsi_exit`: `{"stages": [1.0], "exit_kind": "rsi"}`

In [ ]:
RUN_VARIANTS = False
if RUN_VARIANTS:
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Vol Panics'
    symbols = ["VXX"]  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['vol_double'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[0.5, 0.5],
        strategy_func=vol_panics_strategy,
        variant='vol_double',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Vol Panics'
    symbols = ["VXX"]  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['vol235'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[0.2, 0.3, 0.5],
        strategy_func=vol_panics_strategy,
        variant='vol235',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Vol Panics'
    symbols = ["VXX"]  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['vol1234'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[0.1, 0.2, 0.3, 0.4],
        strategy_func=vol_panics_strategy,
        variant='vol1234',
    )
    
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Vol Panics'
    symbols = ["VXX"]  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['vol_rsi_exit'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=vol_panics_strategy,
        variant='vol_rsi_exit',
    )


## 4. VXX Trend

**Source readiness:** ready. 

**Universe:** VXX; historical note/product changes disclosed

### Short rules

- **Entry:** MA10 < MA30 AND prior MA10 >= prior MA30
- **Additions:** none
- **Exit:** MA10 > MA30 AND prior MA10 <= prior MA30
- **Default parameters:** `{"ma": "sma"}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'VXX Trend Strategy'
symbols = ["VXX"]  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.vxx_trend'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.SHORT,
    position_steps=[1.0],
    strategy_func=vxx_trend_strategy,
)


### Named variants — optional comparison

Each item below preserves its documented thresholds and scale-in rules. Set `RUN_VARIANTS = True` to run them on the same dates. Keep parameter exploration in training/validation data; do not choose the best variant on a final test period.

- `vxx_ema`: `{"ma": "ema"}`

In [ ]:
RUN_VARIANTS = False
if RUN_VARIANTS:
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'VXX Trend Strategy'
    symbols = ["VXX"]  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['vxx_ema'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.SHORT,
        position_steps=[1.0],
        strategy_func=vxx_trend_strategy,
        variant='vxx_ema',
    )


## 5. Trading New Highs

**Source readiness:** experimental. ConnorsRSI(3,2,100), strict less-than percent rank and HV conventions are explicit assumptions absent in document

**Universe:** Stocks; historical point-in-time universe and delisted membership must be supplied

### Long rules

- **Entry:** close > 5 AND mean(volume,21) > 1000000 AND trailing20 contains high==rolling_max(high,252) AND ConnorsRSI < 15; next-day buy limit close*(1-limit_pct)
- **Additions:** none
- **Exit:** ConnorsRSI > 70
- **Default parameters:** `{"limit_pct": 0.07}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Trading New Highs'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.new_highs'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=new_highs_strategy,
    allow_experimental=True,  # explicit assumptions explained above
)


### Named variants — optional comparison

Each item below preserves its documented thresholds and scale-in rules. Set `RUN_VARIANTS = True` to run them on the same dates. Keep parameter exploration in training/validation data; do not choose the best variant on a final test period.

- `new_highs10`: `{"limit_pct": 0.1}`

In [ ]:
RUN_VARIANTS = False
if RUN_VARIANTS:
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Trading New Highs'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['new_highs10'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.LONG,
        position_steps=[1.0],
        strategy_func=new_highs_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='new_highs10',
    )


## 6. TPS — Time, Price, Scale-in

**Source readiness:** ready.  This book’s calls use preceding-20-session average volume >=250000 as an explicit interpretation of its volume minimum.

**Universe:** Non-leveraged, non-inverse ETFs; original 20 ETF membership not supplied

### Long rules

- **Entry:** close > SMA200 AND two consecutive RSI2 < 25
- **Additions:** 10/20/30/40% planned notional; later worse close than latest actual fill; forbid wrong side of SMA200
- **Exit:** RSI2 > 70
- **Default parameters:** `{"aggressive": false}`

### Short rules

- **Entry:** close < SMA200 AND two consecutive RSI2 > 75
- **Additions:** 10/20/30/40% planned notional; later worse close than latest actual fill; forbid wrong side of SMA200
- **Exit:** RSI2 < 30
- **Default parameters:** `{"aggressive": false}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'HPE TPS — Time, Price, Scale-in'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['tps_volume_long'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[0.1, 0.2, 0.3, 0.4],
    strategy_func=tps_strategy,
    variant='tps_volume_long',
)


In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'HPE TPS — Time, Price, Scale-in'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['tps_volume_short'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.SHORT,
    position_steps=[0.1, 0.2, 0.3, 0.4],
    strategy_func=tps_strategy,
    variant='tps_volume_short',
)


## 7. Terror Gaps — specification blocked

**Source readiness:** blocked. Reference price unspecified; daily bars cannot validate post-open sequence. No executable implementation. The code cell below deliberately returns a blocked status. It must not create an entry price from a completed daily low.

**Universe:** ETFs including leveraged ETFs

### Long rules

- **Entry:** prior ConnorsRSI < 5; observe gap-down open; further percentage pullback reference unresolved
- **Additions:** none
- **Exit:** ConnorsRSI > 70
- **Default parameters:** `{"pullback": 0.01}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Terror Gaps'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['buy_fear.terror_gaps'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=terror_gaps_strategy,
)


## Book performance matrix

One row per strategy, direction and ticker. Skipped rows retain their reason. Rerunning a cell replaces that result in the in-memory comparison; every attempted configuration is still journaled on disk. This table is research output, not a recommendation.

In [ ]:
matrix = performance_matrix(results)
display_performance_matrix(matrix)
output_file = PROJECT / "reports/notebooks" / "buy_the_fear_sell_the_greed_performance.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)
matrix.to_csv(output_file, index=False)
print("Saved performance matrix:", output_file)


## Inspect orders, trades and a vectorbt portfolio

Choose any successful strategy key and ticker from `results`. The returned objects remain available for additional vectorbt analysis. Fees include settled borrow charges; reported equity also deducts outstanding borrow liabilities.

In [ ]:
# Example (uncomment after a successful run):
# batch = next(batch for batch in results.values() if batch.results)
# ticker = next(iter(batch.results))
# portfolio = batch.results[ticker].portfolio
# display(batch.results[ticker].orders)
# display(portfolio.positions.records_readable)
# portfolio.positions.regroup(False)[ticker].closed.plot_pnl().show()


## Research limitations

Missing/duplicate sessions fail validation rather than being filled. VIX/TRIN need auxiliary data; incomplete source specifications remain blocked. Current ticker lists can have survivorship bias; VXX has historical product differences. No raw-price corporate-action ledger, live execution, historical locates, margin liquidation or automatic stop is added. Read `docs/methodology.md` and the source catalog for precise assumptions. Authenticated Alpaca and hosted Colab must be verified with your credentials; local notebook tests use mocked SDK responses, never fabricated market results.